# Tâche 1 - Modèles de langue N-grammes

Cette tâche consiste à construire, avec NLTK, des modèles de langue Ngrammes à partir de titres de chansons et de les utiliser pour accomplir 2 sous-tâches qui sont décrites dans l'énoncé du travail pratique.

Les fichiers à utiliser pour cette tâche :
- *t1_titres_chansons.txt*: il contient une liste de titres de chansons, un par ligne de texte. Vous utilisez ce fichier pour la construction des modèles de langues N-grammes.
- *t1_estimation_test.json*: il contient des listes de candidats, chaque liste contenant un seul titre de chanson connu parmi les candidats.
- *t1_generation_test.json*: il contient des titres incomplets se terminant par des mots masqués.  

Consignes:
- Ne pas modifier les fonctions *load_titles* et *load_tests*.
- Utilisez la variable *models* pour conserver les modèles après entraînement.
- Ne pas modifier la signature des fonctions *train_models* et *complete_title*.
- Des modifications aux signatures de fonctions entraîneront des pénalités dans la correction.
- Vous pouvez ajouter des cellules au *notebook* et ajouter toutes les fonctions utilitaires que vous voulez.

## Section 1 - Lecture des fichiers de données (titres de chansons et tests)

On définit ici 2 fonctions pour lire les fichiers d'entraînement et de test.

In [1]:
import json


titres_fn = "./data/t1_titres_chansons.txt"
test_estimation_fn = './data/t1_estimation_test.json'
test_generation_fn = './data/t1_generation_test.json'


def load_titles(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        raw_lines = f.readlines()
    return [x.strip() for x in raw_lines]


def load_tests(filename):
    with open(filename, 'r', encoding='utf-8') as fp:
        test_data = json.load(fp)
    return test_data

In [2]:
titles = load_titles(titres_fn)

Quelques informations à propos des titres:

In [3]:
print("Nombre de titres pour l'entraînement: {}".format(len(titles)))

Nombre de titres pour l'entraînement: 572


On monte également les 2 fichiers de test pour visualiser leur contenu.

In [4]:
import pandas as pd

def get_dataframe(tests):
    return pd.DataFrame.from_dict(tests, orient='columns', dtype=None, columns=None)

In [5]:
tests_estimation = load_tests(test_estimation_fn)

In [6]:
df = get_dataframe(tests_estimation)
pd.set_option("display.max_colwidth", 120)
display(df)

,Candidates,Song
0,"[i want to hold and love, i want to hold something, i want to hold your hand]",i want to hold your hand
1,"[let it be, let me be, let it roll]",let it be
2,"[i want to know the world, i want to love the way you lie, i want to know what love is]",i want to know what love is
3,"[i will survive, i will be there, i will follow and wait]",i will survive
4,"[i will wait, i will eat, i will break free]",i will wait
5,"[i gotta feeling, i gotta rock you, i gotta a feeling to rock you]",i gotta feeling
6,"[i want to break free, i want to break the mold, i want to break my mold free]",i want to break free
7,"[we will rock you, i will rock you, she will rock you]",we will rock you
8,"[i kissed a girl, i kissed a boy, i kissed nobody]",i kissed a girl
9,"[i believe i can fly, i believe i can rock, i believe i can wake up]",i believe i can fly


In [7]:
tests_generation = load_tests(test_generation_fn)

df = get_dataframe(tests_generation)
pd.set_option("display.max_colwidth", 100)
display(df)

,Masked,Song
0,love ***,love shack
1,i want to hold your ***,i want to hold your hand
2,i want to hold *** ***,i want to hold your hand
3,i want *** *** *** ***,i want to hold your hand
4,let it ***,let it be
5,i will ***,i will survive
6,i gotta ***,i gotta feeling
7,i want to break ***,i want to break free
8,i want to *** ***,i want to break free
9,i want *** *** ***,i want to break free


## Section 2 - Construction des modèles de langue N-grammes.

La fonction ***train_models*** prend en entrée une liste de titres de chansons et construit trois modèles: unigramme, bigramme et trigramme.

Les modèles sont conservés dans ***models***, un dictionnaire Python qui prend la forme

<pre>
{
   1: modele_unigramme,
   2: modele_bigramme,
   3: modele_trigramme
}
</pre>

avec comme clé la valeur N du modèle et comme valeur le modèle construit par NLTK.

In [8]:
##Optional : Analyse de détermination du tokenizer (voir plus de détails dans le rapport)
import random
from collections import Counter
import nltk
from nltk import word_tokenize
nltk.download("punkt_tab", quiet=True)   # ressource requise par word_tokenize

# Échantillon aléatoire -longueur, majuscules
random.seed(0)
print("Échantillon :", random.sample(titles, 20))
print("Titres avec majuscules :", sum(t != t.lower() for t in titles))

# Caractères qui ne sont ni lettres ni espaces (ce que le tokeniseur devra décider)
special_chars = Counter(c for t in titles for c in t if not c.isalpha() and c != " ")
print("Caractères spéciaux :", dict(special_chars))

# Enjeu de chaque caractère : fréquence et présence dans les mots masqués (ce qui est évalué)
masked_words = [w for t in tests_generation for w in t["Song"].split()[-t["Masked"].count("***"):]]
for c in special_chars:
    n_titles = sum(c in t for t in titles)
    print(f"{c!r} : {n_titles} titres ({n_titles / len(titles):.0%}) | "
          f"dans les mots masqués : {sum(c in w for w in masked_words)}/{len(masked_words)}")

# Où se trouvent les mots à apostrophe ? (position fixe = contexte prévisible)
positions = Counter(t.split().index(w) for t in titles for w in t.split() if "'" in w)
print("Mots à apostrophe les plus fréquents :", Counter(w for t in titles for w in t.split() if "'" in w).most_common(4))
print("Position dans le titre (0 = premier mot) :", dict(sorted(positions.items())))

# Comment chaque option découpe les cas problématiques
for t in ["let's have a party", "we didn't start the fire", "your husband, my lover"]:
    print(f"{t!r:28} split: {t.split()} | word_tokenize: {word_tokenize(t)}")

Échantillon : ['need u tonight', "love don't cost a thing", 'tell me your secrets', 'i am a man', 'take me with you', "don't step back", 'love me tender', 'waiting for you to come home', 'hold your dreams', 'reach within me', 'hold it down', "you don't know me", 'stand alone', 'need you like a heartache', 'the dance', 'holding on for life', 'tell me you love me', 'the sound of silence', 'waiting for the world to change', "tell me something i don't know"]
Titres avec majuscules : 0
Caractères spéciaux : {"'": 67, ',': 1}
"'" : 65 titres (11%) | dans les mots masqués : 0/66
',' : 1 titres (0%) | dans les mots masqués : 0/66
Mots à apostrophe les plus fréquents : [("don't", 30), ("let's", 12), ("you're", 10), ("i'm", 3)]
Position dans le titre (0 = premier mot) : {0: 49, 1: 5, 2: 9, 3: 1, 4: 2, 5: 1}
"let's have a party"         split: ["let's", 'have', 'a', 'party'] | word_tokenize: ['let', "'s", 'have', 'a', 'party']
"we didn't start the fire"   split: ['we', "didn't", 'start', 'the', '

In [9]:
import nltk
from nltk import word_tokenize, ngrams
from nltk.lm import Laplace
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends
nltk.download("punkt_tab", quiet=True) # requise par word_tokenize

def tokenize(text):
    # Tokenisation NLTK pour separer les contractions et la ponctuation
    return word_tokenize(text)

In [10]:
models = {}

In [14]:
def train_models(texts):
    """ Vous ajoutez à partir d'ici le code dont vous avez besoin
        pour construire les différents modèles N-grammes.
        Cette fonction doit construire tous les modèles en une seule passe.
    """
    tokenized_titles = [tokenize(t) for t in texts]

    for n in (1, 2, 3):
        # Ajoute <s> et </s>, génère les N-grammes d'ordre 1 à n et le vocabulaire
        train_ngrams, vocab = padded_everygram_pipeline(n, tokenized_titles)
        model = Laplace(n)
        model.fit(train_ngrams, vocab)
        models[n] = model

In [16]:
train_models(titles)

# Vérifications rapides pour la taille du vocabulaire et l'effet du lissage de Laplace
for n, model in models.items():
    print(f"N={n} | vocabulaire: {len(model.vocab)} | <s> présent: {'<s>' in model.vocab}")

print("P(want | i)         =", models[2].score("want", ["i"]))
print("P(hand | hold your) =", models[3].score("hand", ["hold", "your"]))
print("P(zzz  | hold your) =", models[3].score("zzz", ["hold", "your"]), "(inconnu)")

N=1 | vocabulaire: 439 | <s> présent: False
N=2 | vocabulaire: 441 | <s> présent: True
N=3 | vocabulaire: 441 | <s> présent: True
P(want | i)         = 0.022916666666666665
P(hand | hold your) = 0.004464285714285714
P(zzz  | hold your) = 0.002232142857142857 (inconnu)


## Section 3 - Estimation - Choix parmi plusieurs candidats

On évalue chaque candidat avec deux métriques :
- **logprob** : somme des log-probabilités des N-grammes du candidat (plus élevé = meilleur). Pénalise les candidats longs.
- **perplexité** : probabilité normalisée par le nombre de jetons (plus faible = meilleur). Indépendante de la longueur.






In [22]:
def text_to_ngrams(text, n):
    padded = list(pad_both_ends(tokenize(text), n=n))
    return list(ngrams(padded, n))

def logprob(model, text):
    return sum(model.logscore(ng[-1], ng[:-1]) for ng in text_to_ngrams(text, model.order))

def perplexity(model, text):
    return model.perplexity(text_to_ngrams(text, model.order))

def select_candidate_by(candidates, n, metric):
    model = models[n]
    if metric == "logprob":
        scores = [logprob(model, c) for c in candidates]
        best = max(range(len(candidates)), key=lambda i: scores[i])
    else:
        scores = [perplexity(model, c) for c in candidates]
        best = min(range(len(candidates)), key=lambda i: scores[i])
    return candidates[best], scores[best]

SELECTION_METRIC = "perplexity"

In [23]:
def select_candidate(candidates, n=3):
    """
        Cette fonction retourne le candidat retenu ainsi que son score attribué par le modèle.

        *candidates* est la liste de passages de textes obtenu à partir du fichier de test.
        Un seul des passage est un titre de chanson connu.

        Le paramètre n désigne le modèle utilisé.
        1 - unigramme NLTK, 2 - bigramme NLTK, 3 - trigramme NLTK

    """
    return select_candidate_by(candidates, n, SELECTION_METRIC)

In [24]:
train_models(titles)

def evaluate_estimation(tests, n, metric):
    # Exactitude : proportion de tests où le titre connu est retenu
    correct = [select_candidate_by(t["Candidates"], n, metric)[0] == t["Song"] for t in tests]
    return sum(correct) / len(correct), correct

results, details = {}, {}
for metric in ("logprob", "perplexity"):
    for n in (1, 2, 3):
        acc, correct = evaluate_estimation(tests_estimation, n, metric)
        results[(metric, n)] = acc
        details[(metric, n)] = correct

display(pd.Series(results).unstack().rename(columns={1: "unigramme", 2: "bigramme", 3: "trigramme"}))

,unigramme,bigramme,trigramme
logprob,0.45,0.5,0.60
perplexity,0.30,0.5,0.65


In [26]:
# Détail par test (utile pour l'analyse d'erreurs)
rows = []
for i, t in enumerate(tests_estimation):
    row = {"titre": t["Song"], "longueurs": [len(tokenize(c)) for c in t["Candidates"]]}
    for (metric, n), correct in details.items():
        row[f"{metric[:4]}-{n}"] = "✓" if correct[i] else "✗"
    rows.append(row)
display(pd.DataFrame(rows))

,titre,longueurs,logp-1,logp-2,logp-3,perp-1,perp-2,perp-3
0,i want to hold your hand,"[6, 5, 6]",✗,✗,✗,✗,✓,✓
1,let it be,"[3, 3, 3]",✗,✗,✗,✗,✗,✗
2,i want to know what love is,"[6, 8, 7]",✗,✗,✗,✗,✗,✗
3,i will survive,"[3, 4, 5]",✓,✓,✓,✗,✗,✗
4,i will wait,"[3, 3, 4]",✓,✓,✓,✓,✓,✓
5,i gotta feeling,"[4, 5, 8]",✓,✓,✓,✗,✗,✗
6,i want to break free,"[5, 6, 7]",✓,✓,✓,✗,✓,✓
7,we will rock you,"[4, 4, 4]",✗,✗,✗,✗,✗,✗
8,i kissed a girl,"[4, 4, 3]",✗,✗,✗,✓,✓,✓
9,i believe i can fly,"[5, 5, 6]",✓,✓,✓,✗,✗,✓


Présentez ici vos résultats et répondez aux questions formulées dans l'énoncé du travail.

**Analyse de résultats**

| Métrique | Unigramme | Bigramme | Trigramme |
|---|---|---|---|
| Log-probabilité | 0,45 | 0,50 | 0,60 |
| Perplexité | 0,30 | 0,50 | **0,65** |

- **L'ordre N compte** : plus le contexte est long, meilleur est le choix. L'unigramme ignore l'ordre des mots et favorise simplement les mots fréquents. Le trigramme reconnaît des séquences typiques des titres.
- **Log-probabilité vs perplexité** : les deux métriques diffèrent surtout quand les candidats n'ont pas la même longueur (12 tests sur 20). La log-probabilité pénalise chaque mot ajouté et favorise le candidat le plus court. Elle réussit *i will survive* mais échoue sur *i want to hold your hand*. La perplexité corrige ce biais et donne le meilleur résultat avec le trigramme.
- **Limite principale** : le lissage de Laplace. Bien que les 20 titres testés soient tous dans le corpus d'entraînement, le trigramme se trompe quand même 7 fois. Avec Laplace, un trigramme vu une fois n'est qu'environ 2 fois plus probable qu'un trigramme jamais vu (0,0045 contre 0,0023). L'avantage d'avoir déjà vu le titre exact devient donc faible, et la fréquence des mots l'emporte. Exemple : *let me be* bat *let it be*. Dans le corpus, 10 titres commencent par *let me* et seulement 2 par *let it* : le distracteur gagne environ ×4 sur ce mot, alors que le vrai titre ne gagne que ×2 à la fin (*let me be* ne termine jamais un titre). Même phénomène pour *love you* face à *love shack* et *i will rock you* face à *we will rock you*.
- **Cas ambigu** : dans le test dont la réponse attendue est *you are so beautiful*, le distracteur *we are so beautiful* est lui aussi un titre du corpus d'entraînement. Deux candidats sur trois sont donc de vrais titres, et le modèle choisit *we are so beautiful*. Ce n'est pas vraiment une erreur du modèle, mais une ambiguïté du test.

**Conclusion**
- **Meilleure configuration : trigramme + perplexité**, avec 13 titres sur 20 correctement identifiés (0,65), contre environ 1 sur 3 au hasard (0,33). C'est donc la métrique retenue dans `select_candidate`.
- **Le contexte est indispensable** : l'unigramme, qui ignore l'ordre des mots, ne fait pas mieux que le hasard (0,30–0,45). Chaque mot de contexte supplémentaire améliore le choix (0,50 avec le bigramme, 0,60–0,65 avec le trigramme).
- **Le principal frein est le lissage de Laplace** : il réduit trop l'avantage des séquences déjà vues, ce qui explique la plupart des erreurs, alors que tous les titres testés sont dans le corpus.
- **Prudence** : avec seulement 20 tests, un test vaut 5 points. Les petits écarts (0,60 contre 0,65) ne sont pas concluants ; l'écart entre l'unigramme et le trigramme (7 tests) est, lui, net.








## Section 4 - Génération - Compléter les titres de chansons

In [ ]:
import random

SPECIAL_TOKENS = {"<s>", "</s>", "<UNK>"}# jetons qui ne sont pas de vrais mots
GENERATION_STRATEGY = "sampling"         # "sampling" ou "greedy"


def build_context(model, tokenized_seed):
    # Padding à gauche comme à l'entraînement
    return ["<s>"] * (model.order - 1) + list(tokenized_seed)

def candidate_words(model, context):
    """ Mots observés après le contexte (même logique que model.generate de NLTK),
        sans les jetons spéciaux. Si aucun mot n'est trouvé, on se replie sur un contexte plus court.
    """
    ctx = context[-(model.order - 1):] if model.order > 1 else []
    while True:
        words = sorted(w for w in model.context_counts(model.vocab.lookup(ctx)) if w not in SPECIAL_TOKENS)
        if words or not ctx:
            return words, ctx
        ctx = ctx[1:]

def next_word(model, context, strategy, rng):
    words, ctx = candidate_words(model, context)
    probs = [model.score(w, ctx) for w in words] # probabilités lissées (Laplace)
    if strategy == "greedy":
        return words[max(range(len(words)), key=lambda i: probs[i])]
    return rng.choices(words, weights=probs)[0]  # tirage pondéré

In [ ]:
def complete_title(model, tokenized_seed, nb_words, random_seed=46):
    """
        Génère des mots et les ajoute à la fin du texte tokenisé.
        Retourne la liste de mots du titre complet (c.-à-d. le *tokenized_seed* + les *nb_words* générés).

        model: Le modèle N-gramme utilisé
        tokenized_seed: le titre incomplet, sans les masques, qui a été tokenisé. Par exemple: ["le", "soleil"].
        random_seed: utilisé par NLTK pour la génération de nombre aléatoire.

    """
    rng = random.Random(random_seed)
    context = build_context(model, tokenized_seed)
    generated = list(tokenized_seed)
    for _ in range(nb_words):
        word = next_word(model, context, GENERATION_STRATEGY, rng)
        generated.append(word)
        context.append(word)
    return generated


In [ ]:
def evaluate_generation(tests, n, strategy, random_seed=46):
    """ Proportion de mots générés identiques aux mots masqués (comparaison position par position). """
    global GENERATION_STRATEGY
    previous, GENERATION_STRATEGY = GENERATION_STRATEGY, strategy
    hits, total, outputs = 0, 0, []
    for t in tests:
        k = t["Masked"].count("***")
        seed = tokenize(t["Masked"].replace("***", "").strip())
        generated = complete_title(models[n], seed, k, random_seed)[-k:]
        hidden = t["Song"].split()[-k:] # les mots masqués ne contiennent pas de contractions
        hits += sum(g == h for g, h in zip(generated, hidden))
        total += k
        outputs.append(generated)
    GENERATION_STRATEGY = previous
    return hits / total, outputs


gen_results, gen_outputs = {}, {}
for strategy in ("sampling", "greedy"):
    for n in (1, 2, 3):
        acc, outputs = evaluate_generation(tests_generation, n, strategy)
        gen_results[(strategy, n)] = acc
        gen_outputs[(strategy, n)] = outputs

# Le tirage dépend de la graine, moyenne sur 10 graines pour un résultat plus fiable
for n in (1, 2, 3):
    gen_results[("sampling (moy. 10 graines)", n)] = sum(
        evaluate_generation(tests_generation, n, "sampling", s)[0] for s in range(10)) / 10

display(pd.Series(gen_results).unstack().rename(columns={1: "unigramme", 2: "bigramme", 3: "trigramme"}).round(3))

,unigramme,bigramme,trigramme
greedy,0.000,0.197,0.697
sampling,0.000,0.227,0.636
sampling (moy. 10 graines),0.012,0.211,0.679


In [ ]:
# Détail par test pour le trigramme
rows = [{"masqué": t["Masked"],
         "sampling": " ".join(gen_outputs[("sampling", 3)][i]),
         "greedy": " ".join(gen_outputs[("greedy", 3)][i]),
         "vrai titre": t["Song"]} for i, t in enumerate(tests_generation)]
display(pd.DataFrame(rows))

,masqué,sampling,greedy,vrai titre
0,love ***,takes,is,love shack
1,i want to hold your ***,tongue,dreams,i want to hold your hand
2,i want to hold *** ***,your hand,your dreams,i want to hold your hand
3,i want *** *** *** ***,you back to you,to be friends me,i want to hold your hand
4,let it ***,go,be,let it be
5,i will ***,wait,always,i will survive
6,i gotta ***,feeling,feeling,i gotta feeling
7,i want to break ***,free,free,i want to break free
8,i want to *** ***,want me,be friends,i want to break free
9,i want *** *** ***,you back to,to be friends,i want to break free


En nombre de mots (sur 66) : trigramme greedy 46, sampling graine 46 : 42 ; bigramme ≈ 13–15 ; unigramme ≈ 0. Avec le trigramme, 28 titres sur 40 (greedy) et 26 sur 40 (sampling) sont complétés entièrement.

**Analyse**
- **Le contexte fait tout.** L'unigramme ne voit aucun contexte : il propose des mots fréquents au hasard (en greedy, toujours *me*) et ne retrouve presque rien. Le bigramme, qui ne voit qu'un mot, hésite entre trop de suites possibles (≈ 1 mot sur 5). Le trigramme, avec deux mots de contexte, retrouve environ 2 mots sur 3 (*we will rock → you*, *my heart will go → on*).
- **Laplace gêne peu la génération.** Le modèle ne choisit que parmi les mots déjà vus après le contexte ; le lissage change seulement leur poids. C'est pourquoi le trigramme fait mieux ici qu'en estimation.
- **Chaque masque supplémentaire complique la tâche.** Un mot faux devient le contexte du mot suivant, et l'erreur se propage. Trigramme : le 1er mot est juste dans 68–78 % des cas, le 2e dans 65 %, le 3e dans 40 %, le 4e jamais.
- **Les débuts de titres partagés trompent le modèle.** Plusieurs titres commencent pareil (*i want to…*, *i will…*, *love…*). Le modèle génère alors une suite correcte, mais celle d'un autre titre (*i want to hold your dreams* au lieu de *your hand*). Le test *i will \*\*\** apparaît même deux fois avec deux réponses différentes (*survive*, *wait*) : impossible de réussir les deux.
- **Greedy vs sampling.** Le greedy choisit toujours la suite la plus fréquente, souvent la bonne : c'est le meilleur score (0,697). Mais il est figé et répète ses erreurs (*i will → always* dans les deux tests). Le sampling tire au hasard parmi les suites possibles : un peu moins bon en moyenne (0,679) et variable selon la graine (0,636 avec la graine 46).

**Conclusion**
WIP

## Section 5 - Partie réservée pour faire nos tests lors de la correction

Merci de ne pas modifier ni retirer cette section du notebook.  